In [75]:
%pip install pandas scikit-learn plotly nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [76]:
from pathlib import Path
import json
import pandas as pd

# Localiza o CSV executando a partir da raiz ou da pasta do notebook.
caminho_csv = Path('_DadosLimpos/jogos_steam.csv')
if not caminho_csv.is_file():
    caminho_csv = Path('..') / caminho_csv

df = pd.read_csv(caminho_csv, encoding='utf-8')
# Converte o texto JSON do CSV em listas de tokens no DataFrame.
df['description_tokens_sem_stopwords'] = (
    df['description_tokens_sem_stopwords'].fillna('[]').map(json.loads)
)
df.head()

,appid,name,last_modified,price_change_number,description,genres,user_tags,description_tokens,description_tokens_normalizados,description_idioma,description_tokens_sem_stopwords
0,10,Counter-Strike,1745368572,37149137,Play the world's number 1 online action game. ...,Action,Action; FPS; Multiplayer; Shooter; Classic; Te...,"[""Play"", ""the"", ""world's"", ""number"", ""1"", ""onl...","[""play"", ""the"", ""world's"", ""number"", ""1"", ""onl...",en,"[play, world's, number, 1, online, action, gam..."
1,20,Team Fortress Classic,1745368565,37149137,One of the most popular online action games of...,Action,Action; FPS; Multiplayer; Classic; Shooter; He...,"[""One"", ""of"", ""the"", ""most"", ""popular"", ""onlin...","[""one"", ""of"", ""the"", ""most"", ""popular"", ""onlin...",en,"[one, popular, online, action, games, time, ,,..."
2,30,Day of Defeat,1745368580,37149137,Enlist in an intense brand of Axis vs. Allied ...,Action,FPS; World War II; Multiplayer; Shooter; Actio...,"[""Enlist"", ""in"", ""an"", ""intense"", ""brand"", ""of...","[""enlist"", ""in"", ""an"", ""intense"", ""brand"", ""of...",en,"[enlist, intense, brand, axis, vs, ., allied, ..."
3,40,Deathmatch Classic,1745368570,37149137,Enjoy fast-paced multiplayer gaming with Death...,Action,Action; FPS; Classic; Multiplayer; Shooter; Fi...,"[""Enjoy"", ""fast-paced"", ""multiplayer"", ""gaming...","[""enjoy"", ""fast-paced"", ""multiplayer"", ""gaming...",en,"[enjoy, fast-paced, multiplayer, gaming, death..."
4,50,Half-Life: Opposing Force,1745368539,37149137,Return to the Black Mesa Research Facility as ...,Action,FPS; Action; Classic; Sci-fi; Singleplayer; Sh...,"[""Return"", ""to"", ""the"", ""Black"", ""Mesa"", ""Rese...","[""return"", ""to"", ""the"", ""black"", ""mesa"", ""rese...",en,"[return, black, mesa, research, facility, one,..."


In [77]:
# Bag of Words: contagem dos termos nas descrições sem stopwords.
from sklearn.feature_extraction.text import CountVectorizer

descricoes_sem_stopwords = df['description_tokens_sem_stopwords']

# Usa os tokens existentes, sem tokenizar o texto novamente.
vectorizer_bow = CountVectorizer(analyzer=lambda tokens: tokens)
X_bow = vectorizer_bow.fit_transform(descricoes_sem_stopwords)

# Mantém a matriz esparsa para economizar memória; linhas seguem a ordem de df.
print(f'Bag of Words: {X_bow.shape[0]} jogos e {X_bow.shape[1]} termos')

df_bow = pd.DataFrame.sparse.from_spmatrix(
    X_bow,
    index=pd.MultiIndex.from_frame(df[['appid', 'name']]),
    columns=vectorizer_bow.get_feature_names_out(),
)
# Exibe 5 jogos e até 20 termos presentes nesses jogos.
colunas_preview = (X_bow[:5].getnnz(axis=0) > 0).nonzero()[0][:20]
df_bow.iloc[:5, colunas_preview]

Bag of Words: 173490 jogos e 121596 termos


,,',(,),",",-,.,/,1,abilities,academy,action,affects,alien,allied,ally,arsenal,arts,assault,assigned,assume
appid,name,,,,,,,,,,,,,,,,,,,,
10,Counter-Strike,0,0,0,0,0,7,0,1,0,0,1,2,0,0,1,0,0,0,0,0
20,Team Fortress Classic,0,0,0,4,4,2,0,0,1,0,1,0,0,0,0,0,0,0,0,0
30,Day of Defeat,0,0,0,2,0,4,2,0,0,0,0,0,0,1,0,1,0,1,0,1
40,Deathmatch Classic,0,1,1,1,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0
50,Half-Life: Opposing Force,2,0,0,1,0,4,0,0,0,1,1,0,1,0,0,0,1,0,1,0


In [78]:
# TF-IDF: ponderação dos termos nas mesmas descrições sem stopwords.
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(analyzer=lambda tokens: tokens)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    df['description_tokens_sem_stopwords']
)

tfidf_palavras = tfidf_vectorizer.get_feature_names_out()

# Mantem os dados esparsos para evitar a alocacao de 157 GiB.
df_tfidf = pd.DataFrame.sparse.from_spmatrix(
    tfidf_matrix,
    columns=tfidf_palavras
).fillna(0.0)

df_tfidf.insert(0, 'appid', df['appid'].values)

# Mostra apenas termos presentes nos primeiros 20 jogos.
colunas_preview_tfidf = (tfidf_matrix[:20].getnnz(axis=0) > 0).nonzero()[0][:20]
preview_tfidf = df_tfidf.iloc[:20, [0] + (colunas_preview_tfidf + 1).tolist()]
from IPython.display import display
with pd.option_context('display.float_format', '{:.4f}'.format):
    display(preview_tfidf)

,appid,!,"""",',(,),",",-,.,/,...,"1,000",10th,12,17,2,2001,360,4,50,:
0,10,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1703,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,20,0.0000,0.0000,0.0000,0.0000,0.0000,0.1209,0.3484,0.0523,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,30,0.0000,0.0000,0.0000,0.0000,0.0000,0.0568,0.0000,0.0982,0.2196,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
3,40,0.0000,0.0000,0.0000,0.1146,0.1145,0.0314,0.0000,0.1359,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,50,0.0000,0.0000,0.1700,0.0000,0.0000,0.0318,0.0000,0.1101,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
5,60,0.0000,0.0000,0.0000,0.0000,0.0000,0.0475,0.0000,0.0411,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
6,70,0.0000,0.0000,0.0000,0.0000,0.0000,0.0367,0.0000,0.0635,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1896,0.0000
7,80,0.0000,0.0000,0.0000,0.0000,0.0000,0.1219,0.0000,0.0263,0.0000,...,0.0000,0.0000,0.1574,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0845
8,130,0.0000,0.0000,0.0000,0.0000,0.0000,0.0606,0.0000,0.0262,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.2264,0.0000,0.0000,0.0000,0.0000
9,220,0.0000,0.0000,0.0000,0.0000,0.0000,0.0915,0.0000,0.0528,0.0000,...,0.0000,0.0000,0.0000,0.1940,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


In [79]:
# Similaridade do cosseno - Bag of Words: um jogo contra todo o catalogo.
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

# Altere o appid para consultar outro jogo (10 = Counter-Strike).
appid_referencia_bow = 10
top_n_bow = 10
posicoes_bow = (df['appid'] == appid_referencia_bow).to_numpy().nonzero()[0]
if len(posicoes_bow) == 0:
    raise ValueError('O appid de referencia nao foi encontrado no DataFrame.')
posicao_bow = int(posicoes_bow[0])
if X_bow[posicao_bow].nnz == 0:
    raise ValueError('O jogo de referencia nao possui termos para comparar.')

# Gera apenas N valores, evitando uma matriz N x N de todos os pares.
similaridade_bow = cosine_similarity(
    X_bow[posicao_bow], X_bow
).ravel()

df_similaridade_bow = df[['appid', 'name']].copy()
df_similaridade_bow['similaridade_cosseno'] = similaridade_bow
# Exclui o proprio jogo do ranking, preservando a ordem original no DataFrame completo.
recomendacoes_bow = (
    df_similaridade_bow
    .iloc[[i for i in range(len(df)) if i != posicao_bow]]
    .nlargest(top_n_bow, 'similaridade_cosseno')
)
print('Referencia:', df.iloc[posicao_bow]['name'], '| Bag of Words')
with pd.option_context('display.float_format', '{:.4f}'.format):
    display(recomendacoes_bow)


Referencia: Counter-Strike | Bag of Words


,appid,name,similaridade_cosseno
27096,990500,PANTY SLIDE VR,0.7220
36983,1253500,-,0.7220
27032,988330,One minute of death,0.7087
100631,2981750,SUBMARINES 2D,0.7048
22554,859610,S.T.R.E.T.C.H.,0.6915
22095,846770,DYSMANTLE,0.6888
31761,1117160,Minds of Nations,0.6869
65928,2010660,Bodacious Agents,0.6860
87055,2598290,P.R.O.T.O.C.O.O.L.,0.6815
85555,2553700,Basilisk looks into the Mirror,0.6807


In [80]:
# Similaridade do cosseno - TF-IDF: um jogo contra todo o catalogo.
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

# Altere o appid para consultar outro jogo (10 = Counter-Strike).
appid_referencia_tfidf = 10
top_n_tfidf = 10
posicoes_tfidf = (df['appid'] == appid_referencia_tfidf).to_numpy().nonzero()[0]
if len(posicoes_tfidf) == 0:
    raise ValueError('O appid de referencia nao foi encontrado no DataFrame.')
posicao_tfidf = int(posicoes_tfidf[0])
if tfidf_matrix[posicao_tfidf].nnz == 0:
    raise ValueError('O jogo de referencia nao possui termos para comparar.')

# Gera apenas N valores, evitando uma matriz N x N de todos os pares.
similaridade_tfidf = cosine_similarity(
    tfidf_matrix[posicao_tfidf], tfidf_matrix
).ravel()

df_similaridade_tfidf = df[['appid', 'name']].copy()
df_similaridade_tfidf['similaridade_cosseno'] = similaridade_tfidf
# Exclui o proprio jogo do ranking, preservando a ordem original no DataFrame completo.
recomendacoes_tfidf = (
    df_similaridade_tfidf
    .iloc[[i for i in range(len(df)) if i != posicao_tfidf]]
    .nlargest(top_n_tfidf, 'similaridade_cosseno')
)
print('Referencia:', df.iloc[posicao_tfidf]['name'], '| TF-IDF')
with pd.option_context('display.float_format', '{:.4f}'.format):
    display(recomendacoes_tfidf)


Referencia: Counter-Strike | TF-IDF


,appid,name,similaridade_cosseno
64071,1961210,Ai-(Onic),0.2767
44515,1451770,Outranked,0.2718
21114,818780,Scrunk,0.2303
16880,705800,Astroe,0.2183
56360,1759550,Conquer the Dungeon,0.1964
67116,2067200,Blobble Wars 2,0.1957
172777,5182970,Objecipher,0.1941
131076,3854150,REDACT,0.1938
110996,3270510,Rescue Q,0.1931
65455,1997290,ParaPerspective,0.1789


In [81]:
# Popularidade = total de avaliacoes positivas + negativas (SteamSpy/Steam).
# Fontes: https://steamspy.com/api.php e https://partner.steamgames.com/doc/store/getreviews
# Ranking restrito aos jogos do CSV com contagem disponivel no cache.
import json
from pathlib import Path
import plotly.graph_objects as go
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

caminho_popularidade = caminho_csv.parent / 'popularidade_steamspy.json'
if not caminho_popularidade.is_file():
    raise FileNotFoundError('Execute 3_Analise_Clusters/coletar_popularidade.py para gerar o cache.')
cache_popularidade = json.loads(caminho_popularidade.read_text(encoding='utf-8'))
popularidade = pd.DataFrame(cache_popularidade['jogos'].values())
if 'fonte' not in popularidade:
    popularidade['fonte'] = cache_popularidade['fonte']
else:
    popularidade['fonte'] = popularidade['fonte'].fillna(cache_popularidade['fonte'])
if popularidade.empty:
    raise ValueError('O cache nao possui contagens de avaliacoes.')

# Posicao explicita para manter o alinhamento com as matrizes vetorizadas.
base_generos = df[['appid', 'name', 'genres']].copy()
base_generos['posicao_matriz'] = range(len(df))
base_generos['genero'] = base_generos['genres'].fillna('').str.split(';')
base_generos = base_generos.explode('genero')
base_generos['genero'] = base_generos['genero'].str.strip()
base_generos = base_generos[base_generos['genero'].ne('')].drop_duplicates(['genero', 'appid'])
base_generos = base_generos.merge(popularidade, on='appid', how='left', validate='many_to_one')

# Nao preenche contagens desconhecidas com zero.
cobertura_generos = base_generos.groupby('genero').agg(
    jogos_no_csv=('appid', 'size'), jogos_com_contagem=('total_avaliacoes', 'count')
)
cobertura_generos['sem_contagem'] = cobertura_generos['jogos_no_csv'] - cobertura_generos['jogos_com_contagem']
print('Cobertura da fonte externa (contagens ausentes ficam fora do ranking):')
display(cobertura_generos)

# Considera cada genero do jogo; desempata pelo appid.
top_10_por_genero = (
    base_generos.dropna(subset=['total_avaliacoes'])
    .sort_values(['genero', 'total_avaliacoes', 'appid'], ascending=[True, False, True])
    .groupby('genero', sort=False).head(10).copy()
)
top_10_por_genero['total_avaliacoes'] = top_10_por_genero['total_avaliacoes'].astype('int64')
top_10_por_genero['posicao_no_genero'] = top_10_por_genero.groupby('genero').cumcount() + 1
print('Ate 10 jogos por genero, conforme a cobertura da fonte:')
display(top_10_por_genero[['genero', 'posicao_no_genero', 'appid', 'name', 'total_avaliacoes', 'coletado_em_utc', 'fonte']])

# Cada jogo aparece uma unica vez nos mapas, mesmo se integrar varios rankings.
jogos_amostra = top_10_por_genero.drop_duplicates('appid').copy()
if jogos_amostra.empty:
    raise ValueError('Nenhum jogo com contagem disponivel para a visualizacao.')
posicoes_amostra = jogos_amostra['posicao_matriz'].to_numpy(dtype=int)
quantidade_jogos = len(jogos_amostra)
rotulos_amostra = [f'{nome} ({appid})' for appid, nome in jogos_amostra[['appid', 'name']].itertuples(index=False, name=None)]
print(f'{quantidade_jogos} jogos unicos selecionados; mesma ordem nos dois mapas.')

# Mesma escala (0 a 1) nos dois graficos para comparar as intensidades.
def plotar_similaridade(matriz, titulo):
    fig = go.Figure(go.Heatmap(
        z=matriz.to_numpy(),
        x=matriz.columns.tolist(),
        y=matriz.index.tolist(),
        colorscale='Blues',
        zmin=0,
        zmax=1,
        texttemplate='%{z:.2f}' if len(matriz) <= 20 else '',
        textfont=dict(size=10),
        colorbar=dict(title='Similaridade'),
        hovertemplate=(
            'Jogo da linha: %{y}<br>'
            'Jogo da coluna: %{x}<br>'
            'Similaridade: %{z:.4f}<extra></extra>'
        ),
    ))
    fig.update_layout(
        title=titulo,
        template='plotly_white',
        width=1200,
        height=1100,
        margin=dict(l=280, r=100, t=90, b=300),
        xaxis=dict(tickangle=-60, tickfont=dict(size=10), automargin=True),
        yaxis=dict(autorange='reversed', tickfont=dict(size=10), automargin=True),
    )
    fig.show()


Cobertura da fonte externa (contagens ausentes ficam fora do ranking):


,jogos_no_csv,jogos_com_contagem,sem_contagem
genero,,,
360 Video,1,1,0
Accounting,6,6,0
Action,73305,33459,39846
Adventure,69188,30713,38475
Animation & Modeling,24,24,0
Audio Production,14,14,0
Casual,72606,31373,41233
Design & Illustration,28,28,0
Documentary,1,1,0


Ate 10 jogos por genero, conforme a cobertura da fonte:


,genero,posicao_no_genero,appid,name,total_avaliacoes,coletado_em_utc,fonte
126249,360 Video,1,1434230,CAT SUDOKU🐱,6,2026-09-16T13:33:34.859966+00:00,https://store.steampowered.com/appreviews/1434...
128348,Accounting,1,1454060,Shooty,14,2026-09-16T13:32:02.980436+00:00,https://steamspy.com/api.php
61909,Accounting,2,849690,Ghost Mountain Roller Coaster,8,2026-09-16T13:31:51.948431+00:00,https://steamspy.com/api.php
385345,Accounting,3,3896710,Sinbadexpress,7,2026-09-16T13:34:27.489155+00:00,https://store.steampowered.com/appreviews/3896...
126233,Accounting,4,1434230,CAT SUDOKU🐱,6,2026-09-16T13:33:34.859966+00:00,https://store.steampowered.com/appreviews/1434...
...,...,...,...,...,...,...,...
30861,Violent,9,559680,Vampire: The Masquerade - Redemption,875,2026-09-16T13:32:07.112296+00:00,https://steamspy.com/api.php
9494,Violent,10,332790,Darkness Assault,655,2026-09-16T13:32:07.112296+00:00,https://steamspy.com/api.php
120708,Web Publishing,1,1384890,Flag Collection,109,2026-09-16T13:32:09.110398+00:00,https://steamspy.com/api.php
126242,Web Publishing,2,1434230,CAT SUDOKU🐱,6,2026-09-16T13:33:34.859966+00:00,https://store.steampowered.com/appreviews/1434...


156 jogos unicos selecionados; mesma ordem nos dois mapas.


In [82]:
# Calcula somente os pares dos jogos selecionados por popularidade.
# Reutiliza a vetorizacao do catalogo completo, sem recalcular o vocabulario ou IDF.
similaridade_amostra_bow = cosine_similarity(X_bow[posicoes_amostra])
df_similaridade_amostra_bow = pd.DataFrame(
    similaridade_amostra_bow, index=rotulos_amostra, columns=rotulos_amostra
)
plotar_similaridade(
    df_similaridade_amostra_bow,
    f'Bag of Words - similaridade entre {quantidade_jogos} jogos dos rankings por genero (avaliacoes SteamSpy/Steam)'
)


In [83]:
# Calcula somente os pares dos jogos selecionados por popularidade.
# Reutiliza a vetorizacao do catalogo completo, sem recalcular o vocabulario ou IDF.
similaridade_amostra_tfidf = cosine_similarity(tfidf_matrix[posicoes_amostra])
df_similaridade_amostra_tfidf = pd.DataFrame(
    similaridade_amostra_tfidf, index=rotulos_amostra, columns=rotulos_amostra
)
plotar_similaridade(
    df_similaridade_amostra_tfidf,
    f'TF-IDF - similaridade entre {quantidade_jogos} jogos dos rankings por genero (avaliacoes SteamSpy/Steam)'
)
